In [0]:
# Get detailed information about the Delta table --> To make sure the table is created in delta format
detail_df = spark.sql("DESCRIBE DETAIL workspace.google_drive.restaurant_1")
display(detail_df)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,41643d29-1b75-4402-b7fe-d66f4438084d,workspace.google_drive.restaurant_1,null,,2026-05-11T19:33:16.399Z,2026-05-11T19:33:46.000Z,List(),List(),1,17846154,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.checkpoint.writeStatsAsStruct -> true, delta.enableRowTracking -> true, delta.checkpoint.writeStatsAsJson -> false, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-67b1ba82-deb8-43cc-a0be-ab11782cef8b, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-ba6c58a1-49f8-4dc3-ad7e-6960ded248ee)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Descriptive statistics for excel & json files to make sure they follows the same strategy

In [0]:
from pyspark.sql.functions import mean, stddev, expr, col

# Filter data for restaurant_1
restaurant_1_df = final_df.filter(col("source_file_name") == "workspace.google_drive.restaurant_1")

# Select numeric columns
numeric_cols = ["price", "quantity", "discount", "total_amount", "rating", "hour"]

# Calculate statistics for each numeric column
print("Descriptive Statistics for restaurant_1:")
print("=" * 80)

for column in numeric_cols:
    stats = restaurant_1_df.select(
        mean(col(column)).alias("mean"),
        expr(f"percentile_approx({column}, 0.5)").alias("median"),
        stddev(col(column)).alias("std")
    ).collect()[0]
    
    print(f"\n{column}:")
    print(f"  Mean:   {stats['mean']:.2f}" if stats['mean'] else f"  Mean:   None")
    print(f"  Median: {stats['median']:.2f}" if stats['median'] else f"  Median: None")
    print(f"  Std:    {stats['std']:.2f}" if stats['std'] else f"  Std:    None")

Descriptive Statistics for restaurant_1:

price:
  Mean:   86.38
  Median: 80.95
  Std:    46.25

quantity:
  Mean:   3.21
  Median: 3.00
  Std:    1.43

discount:
  Mean:   0.04
  Median: None
  Std:    0.05

total_amount:
  Mean:   262.28
  Median: 222.32
  Std:    184.40

rating:
  Mean:   3.70
  Median: 4.00
  Std:    0.64

hour:
  Mean:   16.50
  Median: 16.00
  Std:    4.03


In [0]:
from pyspark.sql.functions import mean, stddev, expr, col

# Filter data for restaurant_1
restaurant_1_df = final_df.filter(col("source_file_name") == "workspace.google_drive.restaurant_json_1")

# Select numeric columns
numeric_cols = ["price", "quantity", "discount", "total_amount", "rating", "hour"]

# Calculate statistics for each numeric column
print("Descriptive Statistics for restaurant_1:")
print("=" * 80)

for column in numeric_cols:
    stats = restaurant_1_df.select(
        mean(col(column)).alias("mean"),
        expr(f"percentile_approx({column}, 0.5)").alias("median"),
        stddev(col(column)).alias("std")
    ).collect()[0]
    
    print(f"\n{column}:")
    print(f"  Mean:   {stats['mean']:.2f}" if stats['mean'] else f"  Mean:   None")
    print(f"  Median: {stats['median']:.2f}" if stats['median'] else f"  Median: None")
    print(f"  Std:    {stats['std']:.2f}" if stats['std'] else f"  Std:    None")

Descriptive Statistics for restaurant_1:

price:
  Mean:   86.48
  Median: 81.14
  Std:    46.27

quantity:
  Mean:   3.20
  Median: 3.00
  Std:    1.43

discount:
  Mean:   0.04
  Median: None
  Std:    0.05

total_amount:
  Mean:   262.24
  Median: 222.53
  Std:    184.27

rating:
  Mean:   3.70
  Median: 4.00
  Std:    0.64

hour:
  Mean:   16.50
  Median: 16.00
  Std:    4.03


Data Types Transformation for safe union

In [0]:
from pyspark.sql.functions import col, lit, to_timestamp, coalesce, cast

#general function to standardize the schema of bronze tables
def standardize_sales(df, table_name):
    return df.select(
        col("order_id").cast("bigint"),
        col("order_date").cast("date"),
        col("hour").cast("int"),
        col("category").cast("string"),
        col("item_name").cast("string"),
        col("price").cast("double"),
        col("quantity").cast("bigint"),
        col("discount").cast("double"),
        col("total_amount").cast("double"),
        col("branch").cast("string"),
        col("payment_method").cast("string"),
        col("order_type").cast("string"),
        col("customer_id").cast("bigint"),
        col("rating").cast("int"),
        col("is_weekend").cast("boolean"), 
        
        # Add column for source file name
        lit(table_name).alias("source_file_name"),
        # Add column for ingestion timestamp
        to_timestamp(lit(None)).alias("silver_processed_at") 
    )

In [0]:

list_of_tables = [
   "workspace.google_drive.restaurant_1","workspace.google_drive.restaurant_2","workspace.google_drive.restaurant_3","workspace.google_drive.restaurant_4","workspace.google_drive.restaurant_5","workspace.google_drive.restaurant_6","workspace.google_drive.restaurant_7","workspace.google_drive.restaurant_json_1","workspace.google_drive.restaurant_json_2"
]

final_df = None

for t_name in list_of_tables:
    #read each table
    temp_df = spark.table(t_name)
    #standarize
    temp_df = standardize_sales(temp_df, t_name)
    #merge
    if final_df is None:
        final_df = temp_df
    else:
        final_df = final_df.unionByName(temp_df)

**Data Exploring & cleaning**

In [0]:
final_df.head()

Row(order_id=922676, order_date=datetime.date(2025, 4, 13), hour=23, category='مشويات', item_name='كباب', price=142.23, quantity=5, discount=0.1, total_amount=640.02, branch='الإسكندرية', payment_method='Cash', order_type='Dine-in', customer_id=88724, rating=3, is_weekend=True, source_file_name='workspace.google_drive.restaurant_1', silver_processed_at=None)

In [0]:
final_df.count()

11110000

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count null values in each column
final_df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in final_df.columns]).show()
#no null values

+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+----------------+-------------------+
|order_id|order_date|hour|category|item_name|price|quantity|discount|total_amount|branch|payment_method|order_type|customer_id|rating|is_weekend|source_file_name|silver_processed_at|
+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+----------------+-------------------+
|       0|         0|   0|       0|        0|    0|       0|       0|           0|     0|             0|         0|          0|     0|         0|               0|           11110000|
+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+----------------+-------------------+



In [0]:
# Count duplicates by order_id
total_rows = final_df.count()
unique_order_ids = final_df.select("order_id").distinct().count()
duplicate_count = total_rows - unique_order_ids

print(f"Total rows: {total_rows}")
print(f"Unique order_ids: {unique_order_ids}")
print(f"Duplicate rows: {duplicate_count}")

Total rows: 11110000
Unique order_ids: 2500000
Duplicate rows: 8610000


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import count, col
from pyspark.sql import functions as F

# Method 1: Find completely duplicate rows (all columns identical)
total_rows = final_df.count()
unique_rows = final_df.distinct().count()
fully_duplicate_count = total_rows - unique_rows

print(f"Total rows: {total_rows}")
print(f"Unique rows (distinct across all columns): {unique_rows}")
print(f"Fully duplicated rows: {fully_duplicate_count}")
print()

# Method 2: Show the duplicate rows with their occurrence count
if fully_duplicate_count > 0:
    # Add a count for each unique combination of all columns
    window_spec = Window.partitionBy(*final_df.columns)
    df_with_count = final_df.withColumn("occurrence_count", F.count("*").over(window_spec))
    
    # Filter to show only rows that appear more than once
    duplicated_rows = df_with_count.filter(col("occurrence_count") > 1).orderBy(col("occurrence_count").desc())
    
    print(f"\nShowing duplicated rows (sorted by occurrence count):")
    display(duplicated_rows)
else:
    print("No fully duplicated rows found!")

Total rows: 11110000
Unique rows (distinct across all columns): 11110000
Fully duplicated rows: 0

No fully duplicated rows found!


In [0]:
duplicate_check = final_df.groupBy("order_id", "item_name").count().filter("count > 1")
duplicate_check.show()

+--------+------------+-----+
|order_id|   item_name|count|
+--------+------------+-----+
|  946481|محشي ورق عنب|    2|
|  964407|   شيش طاووق|    2|
|  972342|       طحينة|    2|
|  673383|  عصير مانجو|    3|
|  715114|    عصير قصب|    2|
|  306593|  عصير مانجو|    4|
|    7634|         شاي|    2|
|   15569|       طحينة|    3|
|   31439|        كفتة|    2|
|   33495|       طحينة|    2|
|   41430|   طاجن فراخ|    2|
|  858533|  عصير مانجو|    3|
|  874403|   شيش طاووق|    2|
|  892329|   طاجن فراخ|    2|
|  593370|        كباب|    2|
|  617175|        كباب|    2|
|  627166|محشي ورق عنب|    2|
|  635101|       طحينة|    2|
|  643036|   محشي كوسة|    2|
|  650971|   بابا غنوج|    2|
+--------+------------+-----+
only showing top 20 rows


In [0]:
final_df.filter(col("order_id")==946481).show(truncate=False)


+--------+----------+----+--------+------------+------+--------+--------+------------+----------+--------------+----------+-----------+------+----------+-----------------------------------+-------------------+
|order_id|order_date|hour|category|item_name   |price |quantity|discount|total_amount|branch    |payment_method|order_type|customer_id|rating|is_weekend|source_file_name                   |silver_processed_at|
+--------+----------+----+--------+------------+------+--------+--------+------------+----------+--------------+----------+-----------+------+----------+-----------------------------------+-------------------+
|946481  |2021-11-19|19  |محاشي   |محشي ورق عنب|96.13 |3       |0.0     |288.4       |طنطا      |Cash          |Dine-in   |51427      |4     |true      |workspace.google_drive.restaurant_1|NULL               |
|946481  |2025-06-08|10  |محاشي   |محشي باذنجان|75.85 |4       |0.1     |273.04      |طنطا      |Card          |Dine-in   |110145     |3     |true      |workspa

**IMPORTANT OBSERVATION: ORDER ID OVERLAP DUE TO DIFFERENT FILES**
We need to generate unique surrogate key => order_id+source_name

In [0]:
from pyspark.sql.functions import concat, col, lit, md5

final_df = final_df.withColumn(
    "source_order_id", 
    md5(concat(col("source_file_name"), lit("_"), col("order_id").cast("string")))
)


In [0]:
print(final_df.count())
print(final_df.filter(col("order_id").isNotNull()).count())
print(final_df.filter(col("order_id").isNotNull()).dropDuplicates(['source_order_id']).count())

11110000
11110000
11110000


In [0]:
#check if any columns categorical data needs edits(trim, lower,same meaning but different naming,...)
print(final_df.select('item_name').distinct().collect())
print(final_df.select('category').distinct().collect())
print(final_df.select('branch').distinct().collect())
print(final_df.select('payment_method').distinct().collect())


[Row(item_name='كباب'), Row(item_name='محشي باذنجان'), Row(item_name='عصير قصب'), Row(item_name='محشي ورق عنب'), Row(item_name='طاجن بامية'), Row(item_name='شيش طاووق'), Row(item_name='طحينة'), Row(item_name='بابا غنوج'), Row(item_name='عصير مانجو'), Row(item_name='سلطة بلدي'), Row(item_name='طاجن فراخ'), Row(item_name='طاجن ملوخية'), Row(item_name='شاي'), Row(item_name='كفتة'), Row(item_name='محشي كوسة')]
[Row(category='مشويات'), Row(category='محاشي'), Row(category='مشروبات'), Row(category='طواجن'), Row(category='مقبلات')]
[Row(branch='الإسكندرية'), Row(branch='طنطا'), Row(branch='القاهرة'), Row(branch='المنصورة'), Row(branch='أسيوط'), Row(branch='الجيزة')]
[Row(payment_method='Cash'), Row(payment_method='Card'), Row(payment_method='Wallet')]


In [0]:
from pyspark.sql.functions import when, col, lit

final_df = final_df.withColumn("rating", when((col("rating") >= 1) & (col("rating") <= 5), col("rating")).otherwise(lit(None)))
final_df = final_df.filter((col("hour") >= 0) & (col("hour") <= 23))
#we can add double check "hour" for only the restaurant's opening hours

final_df.count()

11110000

In [0]:
final_df = final_df.filter((col("total_amount") > 0) & (col("quantity") > 0))
final_df.count()


11058591

In [0]:
11058591-11110000
 

-51409

In [0]:
final_df.filter((col("quantity") > 500)).show()

+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+----------------+-------------------+
|order_id|order_date|hour|category|item_name|price|quantity|discount|total_amount|branch|payment_method|order_type|customer_id|rating|is_weekend|source_file_name|silver_processed_at|
+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+----------------+-------------------+
+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+----------------+-------------------+



In [0]:
from pyspark.sql.functions import current_timestamp

final_df = final_df.withColumn("silver_processed_at", current_timestamp())

#Save final Data
(final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.google_drive.silver_restaurant_master_sales"))

print("Done! Silver table saved with the actual processing timestamp.")

Done! Silver table saved with the actual processing timestamp.
